# Implementation of a Transformer-Based Model Using PyTorch and Hugging Face Transformers

**Experiment:** Text summarization with prompt engineering (zero-shot, one-shot, and few-shot prompting)

## Aim
To implement a pre-trained Transformer model using PyTorch and Hugging Face Transformers and compare its text-summarization outputs when prompted with zero examples, one example, and multiple examples.

## Learning objectives
- Load a pre-trained sequence-to-sequence Transformer from Hugging Face.
- Generate summaries with PyTorch inference.
- Construct zero-shot, one-shot, and few-shot prompts.
- Compare the resulting summaries.

**Task selected:** Text summarization. The notebook uses `google/flan-t5-small`, an instruction-tuned text-to-text model suitable for demonstrating prompt-based tasks.

## Theory

A **Transformer** uses attention mechanisms to model relationships between tokens. Encoder-decoder models such as T5 encode the input text and generate an output sequence. Hugging Face Transformers provides a common interface for downloading a model, tokenizing input, and generating text; PyTorch performs tensor operations and model inference.

**Prompting approaches**

- **Zero-shot:** Give the model an instruction and the passage, without an example.
- **One-shot:** Include one example of a passage and its desired summary before the new passage.
- **Few-shot:** Include two or more examples to demonstrate the required output style.

Prompting does not retrain or update the model weights; it changes the context provided to the model. Results can vary by model and prompt.

## Requirements

The first run downloads the model from Hugging Face, so an internet connection is required. The small model can run on CPU, though inference may be slower than on a GPU.

Install the required libraries in the next cell. If you are using Google Colab, select **Runtime → Change runtime type → T4 GPU** for optional GPU acceleration.

In [ ]:
%pip -q install transformers sentencepiece torch pandas

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display, Markdown

MODEL_NAME = "google/flan-t5-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("Model and tokenizer loaded successfully.")

## Input passage

The same passage is used for all three prompt styles, making the comparison easier to interpret.

In [ ]:
document = (
    "Urban trees can reduce temperatures in towns and cities by providing shade and releasing water vapour. "
    "Their cooling effect can lower energy demand for air conditioning during hot weather. Trees also absorb some "
    "rainwater, help reduce surface runoff, and provide habitats for birds and insects. However, planting programmes "
    "need careful planning. Tree species should suit the local climate, available space, and water supply. Young trees "
    "require maintenance, and roots must be considered when planting near roads and underground utilities. City planners "
    "can achieve better long-term results by choosing suitable native species, protecting existing trees, and ensuring "
    "that neighbourhoods with little green space also receive investment."
)

print("Input passage:\n")
print(document)

## Helper function

The function tokenizes a prompt, moves the tensors to the selected device, and uses `model.generate()` to produce a summary. `torch.inference_mode()` disables gradient tracking because the model is being used for inference rather than training.

In [ ]:
def generate_summary(prompt: str, max_new_tokens: int = 100) -> str:
    """Generate a summary from a prompt using the loaded T5 model."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

## 1. Zero-shot prompting

No demonstration is given. The model receives only the summarization instruction and the input passage.

In [ ]:
zero_shot_prompt = f"""Summarize the following passage in 2 or 3 concise sentences. Preserve the main ideas.

Passage:
{document}

Summary:"""

zero_shot_summary = generate_summary(zero_shot_prompt)
print("ZERO-SHOT SUMMARY\n")
print(zero_shot_summary)

## 2. One-shot prompting

One passage-summary example is included to demonstrate the expected summary style. The target passage is then supplied as a new task.

In [ ]:
one_shot_prompt = f"""Task: Summarize a passage in 1 or 2 concise sentences.

Example passage: Regular exercise strengthens muscles and supports heart health. It can also improve mood and sleep. A balanced routine should match a person's fitness level.
Example summary: Regular exercise supports physical and mental health, including heart health, mood, and sleep. A suitable routine should match the person's fitness level.

Now summarize this passage in 2 or 3 concise sentences.
Passage: {document}
Summary:"""

one_shot_summary = generate_summary(one_shot_prompt)
print("ONE-SHOT SUMMARY\n")
print(one_shot_summary)

## 3. Few-shot prompting

Two demonstrations are included. They illustrate how to shorten a passage while retaining its most important information.

In [ ]:
few_shot_prompt = f"""Task: Summarize each passage in 1 or 2 concise sentences, keeping only the main points.

Example 1 passage: Bees transfer pollen between flowers, helping many plants reproduce. Habitat loss and pesticide exposure have contributed to declines in some bee populations. Planting diverse flowers and reducing pesticide use can help pollinators.
Example 1 summary: Bees are important pollinators, but habitat loss and pesticides threaten some populations. Diverse flowers and reduced pesticide use can help them.

Example 2 passage: A password manager stores account passwords in an encrypted vault. It can create strong, unique passwords and reduce the need to remember each one. Users should protect the vault with a strong master password and multi-factor authentication.
Example 2 summary: Password managers create and securely store unique passwords. A strong master password and multi-factor authentication help protect the vault.

Now summarize the following passage in 2 or 3 concise sentences.
Passage: {document}
Summary:"""

few_shot_summary = generate_summary(few_shot_prompt)
print("FEW-SHOT SUMMARY\n")
print(few_shot_summary)

## Compare the outputs

The following table shows each prompting strategy and its generated summary. Read the summaries for factual coverage, clarity, and conciseness; the comparison is qualitative rather than a benchmark score.

In [ ]:
from IPython.display import display
import pandas as pd

comparison = pd.DataFrame([
    {"Prompting strategy": "Zero-shot", "Summary": zero_shot_summary},
    {"Prompting strategy": "One-shot", "Summary": one_shot_summary},
    {"Prompting strategy": "Few-shot", "Summary": few_shot_summary},
])
display(comparison)

## Result and conclusion

A pre-trained Transformer summarization model was implemented using PyTorch and Hugging Face Transformers. Zero-shot, one-shot, and few-shot prompts were constructed and evaluated on the same passage. Compare the generated summaries to observe how example demonstrations can influence the model's output style and content. The best prompt depends on the task, model, and desired output format.

**Note:** The generated summaries are model outputs, not guaranteed ground truth. Check important facts before using a summary in a consequential setting.